In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import re
import email
from email import policy

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR

In [2]:
df = pd.read_csv(RAW_DATA_DIR / "trec07.csv")
df.shape

(75419, 5)

In [3]:
df.head(2)

,label,subject,email_to,email_from,message
0,1,"Generic Cialis, branded quality@",the00@speedy.uwaterloo.ca,"""Tomas Jacobs"" <RickyAmes@aol.com>",Content-Type: text/html;\nContent-Transfer-Enc...
1,0,Typo in /debian/README,debian-mirrors@lists.debian.org,Yan Morin <yan.morin@savoirfairelinux.com>,"Hi, i've just updated from the gulus and I che..."


In [5]:
df.isnull().sum()

label            0
subject        793
email_to       576
email_from       0
message       1487
dtype: int64

In [6]:
def parse_raw_message(raw: str) -> str:
    """Parse raw MIME email string and extract plain text body."""
    if not isinstance(raw, str) or not raw.strip():
        return ""
    try:
        msg = email.message_from_string(raw, policy=policy.compat32)
    except Exception:
        return raw

    plain_body = ""
    html_body = ""

    if msg.is_multipart():
        for part in msg.walk():
            content_type = part.get_content_type()
            payload = part.get_payload(decode=True)
            if not payload:
                continue
            try:
                decoded = payload.decode("utf-8", errors="ignore")
            except Exception:
                decoded = payload.decode("latin-1", errors="ignore")
            if content_type == "text/plain" and not plain_body:
                plain_body = decoded
            elif content_type == "text/html" and not html_body:
                html_body = decoded
    else:
        payload = msg.get_payload(decode=True)
        if payload:
            try:
                plain_body = payload.decode("utf-8", errors="ignore")
            except Exception:
                plain_body = payload.decode("latin-1", errors="ignore")

    # prefer plain text, fallback to html, then raw
    if plain_body.strip():
        body = plain_body
    elif html_body.strip():
        body = html_body
    else:
        body = raw

    body = re.sub(r"\s+", " ", body).strip()
    return body

In [7]:
df["body"] = df["message"].apply(parse_raw_message)
print(f"Samples after parse: {len(df)}")
print(f"Empty body count: {(df['body'] == '').sum()}")

Samples after parse: 75419
Empty body count: 1487


In [8]:
df.drop(columns=["email_to", "email_from", "message"], inplace=True)
df.columns.tolist()

['label', 'subject', 'body']

In [9]:
# combine subject + body into text, fill empty subject
df["subject"] = df["subject"].fillna("").astype(str).str.strip()
df["body"] = df["body"].fillna("").astype(str)
df["text"] = df["subject"] + " " + df["body"]
df["text"] = df["text"].str.strip()

print(f"Text samples after combine: {len(df)}")
print(f"Empty text count: {(df['text'] == '').sum()}")

Text samples after combine: 75419
Empty text count: 72


In [10]:
df.drop(columns=["subject", "body"], inplace=True)
df.columns.tolist()

['label', 'text']

In [11]:
df["num_urls"]        = df["text"].str.findall(r'https?://\\S+|www\\.\\S+').str.len()
df["num_exclamation"] = df["text"].str.count(r'!')
df["num_question"]    = df["text"].str.count(r'\\?')
df["num_dollar"]      = df["text"].str.count(r'\\$')
df["num_all_caps"]    = df["text"].str.findall(r'\\b[A-Z]{2,}\\b').str.len()
df["num_numbers"]     = df["text"].str.findall(r'\\d+').str.len()
df["word_count"]      = df["text"].str.split().str.len()
caps   = df["text"].str.findall(r'[A-Z]').str.len()
letters = df["text"].str.findall(r'[A-Za-z]').str.len()
df["capital_ratio"]   = np.where(letters > 0, caps / letters, 0)

In [12]:
import emoji
df["emoji_count"] = df["text"].apply(emoji.emoji_count)

In [20]:
df.drop_duplicates(subset="text", inplace=True)
df.shape

(61881, 11)

In [ ]:
from src.preprocess import clean_db_text


In [22]:
df["text"] = df["text"].apply(clean_db_text)
df.drop_duplicates(subset="text", inplace=True)
print(f"After cleaning: {df.shape}")

After cleaning: (60195, 11)


In [23]:
df["text"].iloc[0]

'generic cialis branded quality@ content-type text/html; content-transfer-encoding 7bit do you feel the pressure to perform and not rising to the occasion try v ia gr a your anxiety will be a thing of the past and you will be back to your old self'

In [24]:
df["text"] = df["text"].str.replace("_", "", regex=False)

In [ ]:
from src.preprocess import process_text_advanced
df["text"] = df["text"].apply(process_text_advanced)


In [27]:
# remove - + = from the text
df["text"] = df["text"].str.replace(r'[-+=]{2,}', ' ', regex=True)
# remove > <
df["text"] = df["text"].str.replace(r'[<>/]', '', regex=True)

In [28]:
df["text"].iloc[0]

'gener ciali brand quality@ content-typ texthtml  content-transfer-encod 7bit feel pressur perform rise occas tri v ia gr anxieti thing past back old self'

In [29]:
df.shape

(60195, 11)

In [30]:
df["label"].value_counts()

label
1    36245
0    23950
Name: count, dtype: int64

In [31]:
df.to_csv(PROCESSED_DATA_DIR / "cleaned_trec07.csv", index=False)
print(f"Saved to {PROCESSED_DATA_DIR / 'cleaned_trec07.csv'}")

Saved to C:\Users\aungm\university\computing\cos30049-email-spam-detection\data\processed\cleaned_trec07.csv
